In [73]:
import pandas as pd
import numpy as np
import re

BAD = {"", "[removed]", "[deleted]", "nan", "none", "null"}

def clean_missing_like(s: pd.Series) -> pd.Series:
    s = s.astype(str).fillna("").str.strip()
    s = s.mask(s.str.lower().isin(BAD), "")
    return s

def standardize_title_selftext(df, label, title_col="title", body_col="selftext", author_col="author"):
    out = pd.DataFrame()
    out["author"] = df[author_col].astype(str).fillna("unknown") if author_col in df.columns else "unknown"
    title = clean_missing_like(df[title_col]) if title_col in df.columns else ""
    body  = clean_missing_like(df[body_col])  if body_col in df.columns else ""
    out["text"] = (title + " " + body).str.replace(r"\s+", " ", regex=True).str.strip()
    out["label"] = label
    out = out[out["text"].str.len() > 0].reset_index(drop=True)
    return out

def standardize_text_only(df, label, text_col="text", author_col="author"):
    out = pd.DataFrame()
    out["author"] = df[author_col].astype(str).fillna("unknown") if author_col in df.columns else "unknown"
    out["text"] = clean_missing_like(df[text_col]) if text_col in df.columns else ""
    out["label"] = label
    out = out[out["text"].str.len() > 0].reset_index(drop=True)
    return out

In [74]:
sw = pd.read_csv("swaug22.csv")  # change path if needed

# drop created_utc if present
if "created_utc" in sw.columns:
    sw = sw.drop(columns=["created_utc"])

sw_df = standardize(sw, label="suicidal", title_col="title", body_col="selftext", author_col="author")
sw_df.shape

(13480, 5)

In [75]:
anx_raw  = pd.read_csv("anxiaug22.csv")
dep_raw  = pd.read_csv("depaug22.csv")
lon_raw  = pd.read_csv("loneaug22.csv")   # or "lonely.csv"
norm_raw = pd.read_csv("normal.csv")

# If their column names are different, we’ll adjust in the next cell.

In [76]:
print(norm_raw.shape)
print(norm_raw.columns)
norm_raw.head(3)

(49612, 21)
Index(['Unique_ID', 'text', 'status', 'text_length', 'word_count', 'num_urls',
       'num_emojis', 'num_special_chars', 'num_excess_punct',
       'avg_word_length', 'stopword_ratio', 'type_token_ratio', 'polarity',
       'subjectivity', 'noun_ratio', 'verb_ratio', 'adj_ratio', 'adv_ratio',
       'has_suicidal_keyword', 'has_stress_keyword', 'has_help_keyword'],
      dtype='object')


,Unique_ID,text,status,text_length,word_count,num_urls,num_emojis,num_special_chars,num_excess_punct,avg_word_length,...,type_token_ratio,polarity,subjectivity,noun_ratio,verb_ratio,adj_ratio,adv_ratio,has_suicidal_keyword,has_stress_keyword,has_help_keyword
0,0.0,oh my gosh,Anxiety,0.001678,0.007576,0.0,0.0,0.0,0.0,0.005395,...,1.000000,0.500,0.00,0.0,0.0,0.333333,0.0,False,False,False
1,1.0,"trouble sleeping, confused mind, restless hear...",Anxiety,0.046980,0.034091,0.0,0.0,0.0,0.0,0.015152,...,1.000000,0.350,0.45,0.0,0.1,0.000000,0.0,False,False,False
2,2.0,"All wrong, back off dear, forward doubt. Stay ...",Anxiety,0.058725,0.049242,0.0,0.0,0.0,0.0,0.012200,...,0.926576,0.375,0.45,0.0,0.0,0.071429,0.0,False,False,False


In [77]:
norm_raw = norm_raw[norm_raw["status"].str.lower() == "normal"].reset_index(drop=True)

print("After filtering normal only:", norm_raw.shape)

After filtering normal only: (18391, 21)


In [78]:
norm_raw = norm_raw[["text"]]
norm_raw.shape

(18391, 1)

In [79]:
norm_raw["author"] = ["normal_user_" + str(i) for i in range(len(norm_raw))]
norm_raw.head()

,text,author
0,"Gr gr dreaming of ex crush to be my game, God",normal_user_0
1,wkwkwk what a joke,normal_user_1
2,Leaves are also standby in front of the PC ......,normal_user_2
3,Thank God even though it's just a ride through,normal_user_3
4,wedding teaser concept using the song day6 - o...,normal_user_4


In [80]:
print(norm_raw.columns)
print(norm_df.shape)

Index(['text', 'author'], dtype='object')
(8000, 3)


In [81]:
norm_df = norm_raw.sample(n=8000, random_state=42)

In [82]:
# If your suicidewatch dataframe is called df
sw_df = standardize_title_selftext(sw, label="suicidal", title_col="title", body_col="selftext", author_col="author")
sw_df.head(), sw_df.shape

(           author                                               text     label
 0  MyMapIsTheVOID  maybe I should do it tonight nothing good is c...  suicidal
 1     MikeyMyerss                            My body is betraying me  suicidal
 2    CalmFeed6794                               I’m scared of trying  suicidal
 3      saiffering  I have debt worth 45 grand i dont know what to...  suicidal
 4       Dollsie04  feeling extremely suicidal I'm in a bad relati...  suicidal,
 (13480, 3))

In [83]:
anx_df = standardize_title_selftext(anx_raw, label="anxiety")
dep_df = standardize_title_selftext(dep_raw, label="depression")
lon_df = standardize_title_selftext(lon_raw, label="loneliness")

for name, d in [("anxiety", anx_df), ("depression", dep_df), ("loneliness", lon_df)]:
    print(name, d.shape)

anxiety (7650, 3)
depression (11991, 3)
loneliness (5831, 3)


In [84]:
norm_df = standardize_text_only(norm_df, label="normal", text_col="text", author_col="author")
print(norm_df.shape)
print(norm_df.head())

(8000, 3)
              author                                               text  \
0  normal_user_17887  i am sad that  is  everything  to  see  here  ...   
1   normal_user_3189                      Mashallah, Allahuma Barik ™ ¾   
2  normal_user_11015  @PeoplemapsJulie I am looking for the berocca ...   
3  normal_user_13607      motivated to sleep but i m feeling quite icky   
4  normal_user_11662  some people tweets a lot.ThatÃ¯Â¿Â½s cool...I ...   

    label  
0  normal  
1  normal  
2  normal  
3  normal  
4  normal  


In [85]:
all_df = pd.concat([sw_df, anx_df, dep_df, lon_df, norm_df], ignore_index=True)

print(all_df.shape)
print(all_df["label"].value_counts())
all_df.head(3)

(46952, 3)
label
suicidal      13480
depression    11991
normal         8000
anxiety        7650
loneliness     5831
Name: count, dtype: int64


,author,text,label
0,MyMapIsTheVOID,maybe I should do it tonight nothing good is c...,suicidal
1,MikeyMyerss,My body is betraying me,suicidal
2,CalmFeed6794,I’m scared of trying,suicidal


In [86]:
min_n = all_df["label"].value_counts().min()
print("Balancing to:", min_n)

balanced_df = (all_df.groupby("label", group_keys=False)
               .apply(lambda x: x.sample(min_n, random_state=42))
               .reset_index(drop=True))

balanced_df["label"].value_counts()

Balancing to: 5831


C:\Users\jaisw\AppData\Local\Temp\ipykernel_2284\3368985131.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min_n, random_state=42))


label
anxiety       5831
depression    5831
loneliness    5831
normal        5831
suicidal      5831
Name: count, dtype: int64

In [87]:
before = len(balanced_df)
balanced_df = balanced_df.drop_duplicates(subset=["text"]).reset_index(drop=True)
after = len(balanced_df)
print("Removed duplicates:", before - after)
print("Final:", balanced_df.shape)

Removed duplicates: 637
Final: (28518, 3)


In [88]:
print(anx_raw.columns)
print(dep_raw.columns)
print(lon_raw.columns)

Index(['Unnamed: 0', 'author', 'created_utc', 'score', 'selftext', 'subreddit',
       'title', 'timestamp'],
      dtype='object')
Index(['Unnamed: 0', 'author', 'created_utc', 'score', 'selftext', 'subreddit',
       'title', 'timestamp'],
      dtype='object')
Index(['Unnamed: 0', 'author', 'created_utc', 'score', 'selftext', 'subreddit',
       'title', 'timestamp'],
      dtype='object')


In [89]:
print(sw_df.columns)
print(anx_df.columns)
print(dep_df.columns)
print(lon_df.columns)
print(norm_df.columns)

Index(['author', 'text', 'label'], dtype='object')
Index(['author', 'text', 'label'], dtype='object')
Index(['author', 'text', 'label'], dtype='object')
Index(['author', 'text', 'label'], dtype='object')
Index(['author', 'text', 'label'], dtype='object')


In [91]:
balanced_df["author"].nunique()

22622

In [92]:
# Final sanity check
print(balanced_df.columns)
print(balanced_df["label"].value_counts())

# Save
balanced_df.to_csv("balanced_mental_health_dataset.csv", index=False)
print("Saved as balanced_mental_health_dataset.csv")

Index(['author', 'text', 'label'], dtype='object')
label
anxiety       5785
normal        5783
depression    5737
suicidal      5665
loneliness    5548
Name: count, dtype: int64
Saved as balanced_mental_health_dataset.csv
